In [ ]:
#!pip --version

pip 26.2.1 from D:\yujin0902\ex0914\.venv\Lib\site-packages\pip (python 3.12)



In [21]:
#!pip install dotenv
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
#단순 프롬포트 틀(하나의 텍스트 툴)
from langchain_core.prompts import PromptTemplate
#대화 구조/역할 나누는 프롬포트 툴 (system/user 등 메시지 역할을 구분)
#from langchain_core.prompts import ChatPromptTemplate
#output parser -> llm 답변을 json형태로 쉽게 만들어주는
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain_teddynote import logging



In [3]:
import os

print(os.getenv("LANGSMITH_ENDPOINT"))

https://api.smith.langchain.com


In [4]:
llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4o-mini"
)

template = "{country}의 수도는 어디인가요?"

prompt = PromptTemplate.from_template(template)
#prompt = prompt.format(country = "미국")
#prompt
chain = prompt | llm

chain.invoke("미국").content

'미국의 수도는 워싱턴 D.C.입니다.'

In [23]:
llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4o-mini"
)

template = "{country}의 수도는 어디인가요?"

prompt = PromptTemplate(
    template=template,
    input_variables=["country"],
)

prompt.format(country="영국")

'영국의 수도는 어디인가요?'

In [7]:
template = "{country1}과 {country2}의 수도는 각각 어디인가요?"

prompt = PromptTemplate(
    template=template,
    input_variables=["country1"],
    partial_variables={
        "country2":"미국"
    },
)


#prompt.format(country1="영국")
#country2에 캐나다를 미리 채워넣는. 하나를 미리 확정하는
#prompt_partial = prompt.partial(country2 = "캐나다")
#prompt_partial

chain = prompt | llm
#아직 값이 지정되지 않는 country1에 들어감
#chain.invoke("대한민국").content

chain.invoke({"country1": "체코", "country2": "호주"})

AIMessage(content='체코의 수도는 프라하(Prague)이고, 호주의 수도는 캔버라(Canberra)입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 19, 'total_tokens': 47, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_62dcbff70a', 'id': 'chatcmpl-EOe7wJG6h0l752ewPFGtUfNtl3npC', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0a910-8619-7870-8a64-072eb3ccf0c3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19, 'output_tokens': 28, 'total_tokens': 47, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasonin

## 부분변수 활용하기
- 함수를 사용하여 프롬포트 일부를 미리 설정

In [59]:
from datetime import datetime

#오늘 날짜를 출력
#datetime.now().strftime("%B %d")

def get_today():
    return datetime.now().strftime("%B %d")

#today에 값을 넣지 않아도 함수가 자동으로 값을 채워줌
prompt = PromptTemplate(
    template="오늘 날짜는 {today}입니다. 오늘이 생일인 유명인 {n}명을 나열해 주세요. 생년월일을 표기해주세요",
    input_variables=["n"],
    partial_variables={
        "today":get_today #딕셔너리 형태로 partial_variables를 전달
    },
)

#prompt.format(n=3)
chain = prompt | llm
#print(chain.invoke(3).content)
print(chain.invoke({"today":"Jan 02", "n": 3}).content)

오늘, 1월 2일에 생일인 유명인 3명은 다음과 같습니다:

1. **아이작 뉴턴 (Isaac Newton)** - 1643년 1월 4일 (그레고리력 기준)
2. **J.R.R. 톨킨 (J.R.R. Tolkien)** - 1892년 1월 3일
3. **드레이크 (Drake)** - 1986년 10월 24일

참고로, 아이작 뉴턴의 생일은 그레고리력으로 1월 4일이지만, 율리우스력으로는 12월 25일입니다.


## YAML 파일로 부터 프롬포트 템플릿 만들기

In [60]:
from langchain_core.prompts import load_prompt
#llm 결과를 문자열로 정리
from langchain_core.output_parsers import StrOutputParser
#출력 화면을 보여주는 용도, 스트리밍 결과를 화면에 출력
from langchain_teddynote.messages import stream_response

#yaml파일에서 프롬포트 템플릿을 가져옴
prompt = load_prompt("D:/yujin0902/ex0914/fruit_color.yaml", encoding="utf-8")
#prompt.format(fruit="사과")

chain = prompt | llm | StrOutputParser()

answer = chain.stream({"fruit":"오렌지"})
stream_response(answer)

오렌지의 색깔은 주로 주황색입니다. 하지만 오렌지의 종류나 숙성 정도에 따라 색상이 약간 다를 수 있습니다. 일반적으로는 밝은 주황색에서 진한 주황색까지 다양합니다.

Failed to multipart ingest runs: Connection error caused failure to POST https://eu.api.langchain.com/runs/multipart in LangSmith API. Please confirm your LANGCHAIN_ENDPOINT. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'eu.api.langchain.com\', port=443): Max retries exceeded with url: /runs/multipart (Caused by NameResolutionError("HTTPSConnection(host=\'eu.api.langchain.com\', port=443): Failed to resolve \'eu.api.langchain.com\' ([Errno 11001] getaddrinfo failed)"))'))
Content-Length: 7487
API Key: lsv2_********************************************82trace=01a0a812-98fe-7d12-a832-8ccb75d14ca5,id=01a0a812-98fe-7d12-a832-8ccb75d14ca5; trace=01a0a812-98fe-7d12-a832-8ccb75d14ca5,id=01a0a812-98ff-7270-89c2-7076f77b3023; trace=01a0a812-98fe-7d12-a832-8ccb75d14ca5,id=01a0a812-9900-70b3-922a-36dcbb9a4fba
Failed to multipart ingest runs: Connection error caused failure to POST https://eu.api.langchain.com/runs/multipart in LangSmith API. Please confirm your LANGCHAIN_ENDPOINT. Conn

## ChatPromptTemplate
- 대화형 챗봇과 같이 상호작용하는 시스템에서 사용

In [63]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_template("{counrty}의 수도는")
chat_prompt.format(counrty="대한민국")


'Human: 대한민국의 수도는'

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    temperature=0.1,
    model_name = "gpt-4o-mini"
)

chat_template = ChatPromptTemplate.from_messages(
    [
        ("system","당신은 친절한 AI 어시스턴트입니다. 당신의 이름은 {name}입니다."),
        ("human","반가워요"),
        ("ai","안녕하세요, 무엇을 도와드릴까여"),
        ("human","{user_input}"),
    ]
)
#huma에는 user_input을 이용하여 사용자 입력을 받음

#chain을 사용할 경우 필요 없음
# messages = chat_template.format_messages(
#     name = "테디", 
#     user_input = "당신의 이름은 무엇입니까?"
# )

#llm.invoke(messages).content
chain = chat_template | llm
chain.invoke({"name":"Teddy", "user_input":"당신의 이름은 무엇입니까"}).content

'제 이름은 Teddy입니다. 당신과 대화하게 되어 기쁩니다! 어떻게 도와드릴까요?'

Failed to multipart ingest runs: Connection error caused failure to POST https://eu.api.langchain.com/runs/multipart in LangSmith API. Please confirm your LANGCHAIN_ENDPOINT. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'eu.api.langchain.com\', port=443): Max retries exceeded with url: /runs/multipart (Caused by NameResolutionError("HTTPSConnection(host=\'eu.api.langchain.com\', port=443): Failed to resolve \'eu.api.langchain.com\' ([Errno 11001] getaddrinfo failed)"))'))
Content-Length: 9441
API Key: lsv2_********************************************82trace=01a0a824-20fd-7e73-8388-f95a97063f31,id=01a0a824-20fd-7e73-8388-f95a97063f31; trace=01a0a824-20fd-7e73-8388-f95a97063f31,id=01a0a824-2100-7ea3-bd31-a26883a080c3; trace=01a0a824-20fd-7e73-8388-f95a97063f31,id=01a0a824-2102-7291-8821-09cad3f8de86
Failed to multipart ingest runs: Connection error caused failure to POST https://eu.api.langchain.com/runs/multipart in LangSmith API. Please confirm your LANGCHAIN_ENDPOINT. Conn

## MessagesPlaceholder
- 대화에서 확정된 메시지는 아니지만
- 나중에 채워질 메시지를 채우기 위해 임시로 확보한 자리

- 대화 내용을 기록하기 위해 사용

In [ ]:
from langchain_core.output_parsers import StrOutputParser #문자열로 정리
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

chat_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 요약 전문 AI 어시스턴트입니다. 당신의 임무는 주요 키워드로 대화를 요약하는 것입니다.",
        ),
        #실제 대화가 들어갈 자리를 미리 확보
        MessagesPlaceholder(variable_name="conversation"),
        ("human","지금까지의 대화를 {word_count} 단어로 요약합니다."),    
    ]
)

#실제 데이터 전달
formatted_chat_prompt = chat_prompt.format(
    word_count=5,
    conversation=[
        ("human","안녕하세요! 저는 새로 입사한 테디입니다."),
        ("ai","반가워요"),
    ],
)

print(formatted_chat_prompt)

System: 당신은 요약 전문 AI 어시스턴트입니다. 당신의 임무는 주요 키워드로 대화를 요약하는 것입니다.
Human: 안녕하세요! 저는 새로 입사한 테디입니다.
AI: 반가워요
Human: 지금까지의 대화를 5 단어로 요약합니다.


In [11]:
llm = ChatOpenAI(
    temperature=0.1,
    model="gpt-4o-mini"
)

chat_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 요약 전문 AI 어시스턴트입니다. 당신의 임무는 주요 키워드로 대화를 요약하는 것입니다.",
        ),
        #실제 대화가 들어갈 자리를 미리 확보
        MessagesPlaceholder(variable_name="conversation"),
        ("human","지금까지의 대화를 {word_count} 단어로 요약합니다."),    
    ]
)

chain = chat_prompt | llm | StrOutputParser()

chain.invoke(
    {
        "word_count": 20,
        "conversation": [
            (
                "human",
                "안녕하세요! 저는 오늘 새로 입사한 테디입니다. 만나서 반갑습니다.",
            ),
            ("ai","반가워요!"),
        ],
    }
)

'테디가 새로 입사했으며, 인사하며 만나서 반갑다고 전했습니다.'

## Few-shot (퓨삿 프롬프트)
- LLM에게 원하는 답변의 예시를 몇 개 보여준 다음, 새로운 문제를 풀게하는 프롬프트

In [32]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_teddynote import logging
from langchain_openai import ChatOpenAI
from langchain_teddynote.messages import stream_response

llm = ChatOpenAI(
    temperature=0.1,
    name = "gpt-4o-mini"
)

examples = [
    {
        "question": "스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 스티브 잡스는 몇 살에 사망했나요?
중간 답변: 스티브 잡스는 56세에 사망했습니다.
추가 질문: 아인슈타인은 몇 살에 사망했나요?
중간 답변: 아인슈타인은 76세에 사망했습니다.
최종 답변은: 아인슈타인
""",
    },
    {
        "question": "네이버의 창립자는 언제 태어났나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 네이버의 창립자는 누구인가요?
중간 답변: 네이버는 이해진에 의해 창립되었습니다.
추가 질문: 이해진은 언제 태어났나요?
중간 답변: 이해진은 1967년 6월 22일에 태어났습니다.
최종 답변은: 1967년 6월 22일
""",
    },
]

#예시 하나를 어떤 형식으로 보여줄지 정하는 것
example_prompt = PromptTemplate.from_template(
    "Question:\n{question}\nAnswer:\n{answer}"
)

#print(example_prompt.format(**examples[0]))

#
prompt = FewShotPromptTemplate(
    examples=examples, #실제 예시 데이터를 넣는 곳
    example_prompt=example_prompt, #각 예시를 어떤 형식으로 출력할지 알려주는
    suffix="Question:\n{question}\nAnswer:", #앞에 예시들을 참고해서 이 새로운 질문에 대답해봐
    input_variables=["question"], #prompt가 외부에서 받을 수 있는 변수는 question
)

chain = prompt|llm|StrOutputParser()

question = "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"
final_prompt = prompt.format(question=question) #최종 prompt 만들기
print(final_prompt)

# answer=llm.stream(final_prompt)
# stream_response(answer)

answer = chain.stream(
    {
        "question":"구글이 창립된 연도에 bill gates의 나이는 몇살인가요?"
    }
)

stream_response(answer)

Question:
스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 스티브 잡스는 몇 살에 사망했나요?
중간 답변: 스티브 잡스는 56세에 사망했습니다.
추가 질문: 아인슈타인은 몇 살에 사망했나요?
중간 답변: 아인슈타인은 76세에 사망했습니다.
최종 답변은: 아인슈타인


Question:
네이버의 창립자는 언제 태어났나요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 네이버의 창립자는 누구인가요?
중간 답변: 네이버는 이해진에 의해 창립되었습니다.
추가 질문: 이해진은 언제 태어났나요?
중간 답변: 이해진은 1967년 6월 22일에 태어났습니다.
최종 답변은: 1967년 6월 22일


Question:
Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 구글이 창립된 연도는 언제인가요?
중간 답변: 구글은 1998년에 창립되었습니다.
추가 질문: 빌 게이츠는 1998년에 몇 살이었나요?
중간 답변: 빌 게이츠는 1998년에 43세였습니다.
최종 답변은: 43세

## 예제 선택기

### 퓨샷의 단점
- 퓨샷 예제는 모든 예시가 프롬프트에 들어가기 때문에 비용이 많이 듦
- 프롬프트의 길이가 길어질 수록 컨텍스트를 많이 차지해서 모델이 중요한 정보에 집중하기 어려울 수 있음

### 예제 선택기 사용 이유
- 예제선택기는 질문과 유사하거나 관련성 높은 예시만 선택해 프롬프트에 넣음
- 불필요 예시를 줄여 프롬프트 길이와 입력 토큰을 줄여 비용 절감
- 관성성 높은 예시를 제공하여 원하는 답변 형식을 더 효과적으로 유도

In [ ]:
#fewshot에서 어떤 예시를 사용할지 골라주는 예제 선택기 클래스
from langchain_core.example_selectors import (
    MaxMarginalRelevanceExampleSelector, #의미가 비슷한 예씨 선택
    SemanticSimilarityExampleSelector, #비슷하면서도 서로 다른 예시 선택
)
#OpenAIEmbeddings 텍스트를 벡터로 변환해주는 클래스
from langchain_openai import OpenAIEmbeddings
#chroma는 벡터 데이터 베이스
from langchain_chroma import Chroma

#vector db 생성 (저장소 이름, 임베딩 클래스)
chroma = Chroma("example_selector",OpenAIEmbeddings())

example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples,
    OpenAIEmbeddings(),
    Chroma,
    k=1,
)

